# Lab 05: Frameworks and automatic differentiation

## 1. Value node
A scalar Value stores data, gradient, parent nodes and a local derivative rule. Gradients start at zero; leaf nodes have no parents. Implementation: src/value.py.


In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src' / 'value.py').exists() else Path.cwd() / 'lab05'
sys.path.insert(0, str(ROOT.parent))
from lab05.src.value import Value
from lab05.src.experiments import verify_gradients, train_comparison, execution_comparison
import torch
OUT = ROOT / 'results'
OUT.mkdir(exist_ok=True)
torch.set_num_threads(2)
a = Value(2, label='a')
assert a.data == 2.0 and a.grad == 0.0 and a.parents == () and callable(a._backward)
print(a)
print('Parents:', a.parents)
print('Local rule callable:', callable(a._backward))


C:\Users\maria\Desktop\university\ML_73952\lab01\.venv\Lib\site-packages\torch\cuda\__init__.py:56: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Value(data=2.0, grad=0.0)
Parents: ()
Local rule callable: True


## 2. Operations and forward graph
Addition, multiplication, constant powers and ReLU create output nodes linked to their operands. Local derivatives are 1 for addition, the opposite operand for multiplication, p*x**(p-1) for powers, and 1 for positive ReLU inputs or 0 otherwise. The derivative at zero is taken as zero. Sigmoid and log support the BCE training experiment in section 5.


In [2]:
a,b,c = Value(2,label='a'),Value(-3,label='b'),Value(1,label='c')
product = a*b
product.label = 'product'
shifted = product+c
shifted.label = 'shifted'
squared = shifted**2
squared.label = 'squared'
out = squared.relu().sum()
for node in [a,b,c,product,shifted,squared,out]:
    print(node.label or node.op, 'data=', node.data, 'parents=', [p.label or p.op for p in node.parents])
assert out.data == 25
assert product.parents == (a,b)
assert (2+a).data == 4 and (2*a).data == 4
assert Value(-1).relu().data == 0


a data= 2.0 parents= []
b data= -3.0 parents= []
c data= 1.0 parents= []
product data= -6.0 parents= ['a', 'b']
shifted data= -5.0 parents= ['product', 'c']
squared data= 25.0 parents= ['shifted']
+ data= 25.0 parents= ['ReLU', '']


## 3. Backward pass
An iterative depth-first traversal creates a topological order, with every parent before its child. Backward sets the output gradient to 1 and invokes local rules in reverse order. Rules accumulate contributions with +=. Parameter gradients must be cleared before each new training step; training constructs a fresh graph each step.


In [3]:
out.backward()
print('Gradient a:', a.grad)
print('Gradient b:', b.grad)
print('Gradient c:', c.grad)
assert [a.grad,b.grad,c.grad] == [30.0,-20.0,-10.0]
x = Value(3)
shared = x*x
loss = shared+shared+x
order = loss.topological_order()
loss.backward()
assert x.grad == 13.0
print('Shared-branch gradient:', x.grad)
print('Topological nodes:', len(order), '; unique:', len(set(order)))


Gradient a: 30.0
Gradient b: -20.0
Gradient c: -10.0
Shared-branch gradient: 13.0
Topological nodes: 4 ; unique: 4


## 4. Gradient verification
Compare ((a*b+c)**2).relu().sum() against float64 PyTorch autograd and central finite differences with step 1e-5. Four cases include negative inputs and a zero result. A scalar sum is represented as addition of zero. Both maximum absolute errors must be below 1e-6.


In [4]:
gradient_table = verify_gradients(OUT)
print(gradient_table.to_string(index=False))
print('Max PyTorch error:', gradient_table.torch_error.max())
print('Max numerical error:', gradient_table.numerical_error.max())


 case variable  custom  pytorch  numerical  torch_error  numerical_error
    0        a  30.000   30.000     30.000          0.0     6.406218e-10
    0        b -20.000  -20.000    -20.000          0.0     1.310241e-10
    0        c -10.000  -10.000    -10.000          0.0     3.785772e-10
    1        a   1.330    1.330      1.330          0.0     6.128431e-13
    1        b  -2.850   -2.850     -2.850          0.0     1.314060e-12
    1        c   1.900    1.900      1.900          0.0     1.022626e-11
    2        a  -0.024   -0.024     -0.024          0.0     2.009157e-14
    2        b  -0.016   -0.016     -0.016          0.0     2.423756e-14
    2        c  -0.080   -0.080     -0.080          0.0     3.060052e-14
    3        a   0.000    0.000      0.000          0.0     0.000000e+00
    3        b   0.000    0.000      0.000          0.0     0.000000e+00
    3        c   0.000    0.000      0.000          0.0     0.000000e+00
Max PyTorch error: 0.0
Max numerical error: 6.40621

## 5. make_moons training and speed comparison
Use 100 samples, noise 0.2, an 80/20 stratified split, and train-fitted input scaling (seed 42). Both implementations use a 2-8-1 MLP with sigmoid hidden/output activations and mean BCE. Initial weights, data, float64 precision and full-batch SGD are matched. Train 200 epochs at learning rate 1.0 and report the median of five fresh runs. Setup and evaluation are excluded from timing. Parameter gradients at initialization and final weights are checked against PyTorch.


In [5]:
training_runs, training_medians, trained_torch = train_comparison(OUT)
print(training_runs.to_string(index=False))
print('\nMedian of five runs:')
print(training_medians.to_string(index=False))


Training repetition 1/5 finished.
Training repetition 2/5 finished.
Training repetition 3/5 finished.
Training repetition 4/5 finished.
Training repetition 5/5 finished.
 run framework  train_seconds  last_training_loss  test_accuracy
   1    scalar      46.133778            0.285802           0.85
   1   pytorch       0.389677            0.285802           0.85
   2    scalar      38.248674            0.285802           0.85
   2   pytorch       0.304974            0.285802           0.85
   3    scalar      26.377428            0.285802           0.85
   3   pytorch       0.332557            0.285802           0.85
   4    scalar      30.590980            0.285802           0.85
   4   pytorch       0.307560            0.285802           0.85
   5    scalar      28.676086            0.285802           0.85
   5   pytorch       0.551907            0.285802           0.85

Median of five runs:
framework  train_seconds  last_training_loss  test_accuracy
   scalar      30.590980         

## 6. Graph inspection, execution modes and tensor layout
Trace and freeze the trained small PyTorch model with torch.jit.trace. Save the inlined graph, check outputs, then compare eager and traced inference after warm-up at batch sizes 1 and 128. Trace setup is timed separately. All measurements use CPU with two Torch threads.

Inspect two equal 512x512 float64 tensors: a non-contiguous transpose view and its contiguous copy. Record shape, dtype, strides and storage sharing, then time the same matrix multiplication by a 512x128 tensor. Copy creation is excluded from matmul timing and measured separately. No universal speed penalty is assumed.


In [6]:
execution_table, layout_table, copy_ms = execution_comparison(OUT, trained_torch)
print('Traced graph:')
print((OUT / 'traced_graph.txt').read_text())
print('Eager versus traced:')
print(execution_table.to_string(index=False))
print('Tensor metadata and matmul timings:')
print(layout_table.to_string(index=False))
print('Contiguous copy creation (ms):', copy_ms)


Traced graph:
graph(%self.1 : __torch__.torch.nn.modules.container.___torch_mangle_7.Sequential,
      %input.1 : Tensor):
  %self.2.weight : Double(1, 8, strides=[8, 1], requires_grad=0, device=cpu) = prim::Constant[value= 1.9905 -2.4954  0.7782  0.2786  0.6575 -1.8872  0.7264  1.3432 [ CPUDoubleType{1,8} ]]()
  %self.2.bias : Double(1, strides=[1], requires_grad=0, device=cpu) = prim::Constant[value={-0.556073}]()
  %self.0.weight : Double(8, 2, strides=[2, 1], requires_grad=0, device=cpu) = prim::Constant[value=<Tensor>]()
  %self.0.bias : Double(8, strides=[1], requires_grad=0, device=cpu) = prim::Constant[value=-0.1294  0.1974 -0.1164 -0.0147 -0.0328  0.1288 -0.0755 -0.0672 [ CPUDoubleType{8} ]]()
  %input.3 : Tensor = aten::linear(%input.1, %self.0.weight, %self.0.bias), scope: __module.0 # C:\Users\maria\Desktop\university\ML_73952\lab01\.venv\Lib\site-packages\torch\nn\modules\linear.py:116:0
  %input.5 : Tensor = aten::sigmoid(%input.3), scope: __module.1 # C:\Users\maria\Desk

### Interpretation

Tracing can help repeated inference on a stable graph by reducing Python dispatch overhead. Compare the measured per-call saving with trace setup cost; a small number of calls may not amortize that cost. These timings cover tracing/freezing, not torch.compile.

Contiguity effects depend on the operation and strides. A transpose view can be handled efficiently by matrix multiplication kernels. Compare matched inputs and include copy creation cost when deciding whether a contiguous copy is useful. The tables above contain the current run; report.md records the completed experiment.
